# Part 2: Analysis (B)

All analysis cells are read-only (`q(...)` on `gold.*`). The only exception is the cell that saves a mart, labeled `# SAVE MART`. Each question gets a 2-3 sentence conclusion after the query.

## B1. Do late deliveries lower review scores?
- **Tables:** `fact_orders` (A), `fact_reviews`
- **Method:** average `review_score_avg` for late vs on-time orders; share of 1-2 star reviews by delay bucket (on time, 1-3 days late, 4-7 days, 8+ days). Optional: Mann-Whitney test in pandas/scipy.
- **Output:** DAX measures `Avg Review Score` by `is_late`; delay buckets as a calculated column.
- **Conclusion:** _write after running_

## B2. Which sellers perform best and worst?
- **Tables:** `fact_order_items`, `fact_orders`, `fact_reviews`, `dim_sellers`
- **Method:** per seller: revenue, orders, average review score, late-order share, average delivery days. Ignore sellers with very few orders (for example under 10) when ranking.
- **Output:** saved mart `gold.mart_seller_performance`.
- **Conclusion:** _write after running_

## B3. Which product categories drive revenue and satisfaction?
- **Tables:** `fact_order_items`, `dim_products`, `fact_reviews`
- **Method:** revenue, items sold, average price and average review score per `category_name_english`; top 10 by revenue; categories with the lowest scores.
- **Output:** DAX measures by `dim_products[category_name_english]`.
- **Conclusion:** _write after running_

## B4. Does rain affect orders? (São Paulo only)
- **Tables:** `fact_orders` (A), `dim_customers` (A), `dim_date`, `weather`
- **Method:** daily orders and revenue on rainy vs dry days (`precipitation > 0`, or above a chosen threshold); also late-delivery share on rainy vs dry days. Only `state = 'SP'`.
- **Output:** saved mart `gold.mart_daily_sales_weather`.
- **Limitation:** weather covers São Paulo only; mark every visual "SP only".
- **Conclusion:** _write after running_

## B5. Do holidays change sales?
- **Tables:** `fact_orders`, `dim_date`
- **Method:** average daily orders and revenue on holidays vs normal days; also the days just before and after major holidays. Compare weekdays with weekdays to avoid weekend bias.
- **Output:** DAX measures by `dim_date[is_holiday]`.
- **Conclusion:** _write after running_

## B6. (Optional) Does the exchange rate matter?
- **Tables:** `fact_order_items`, `dim_date`
- **Method:** monthly revenue in BRL and in USD (`price * usd_per_brl`); does USD revenue move differently from BRL revenue?
- **Output:** DAX measure `Revenue USD`.
- **Conclusion:** _write after running_

In [27]:
import pandas as pd
from sqlalchemy import text
import sys

sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")
from db import get_engine

engine = get_engine()


## B1

Biz burada sifarişlərin gecikmə vaxtında verilən qiymətləndirməyə təsir edir ya yox ona baxırıq. Nəticə olaraq vaxtında çatdırılmış sifarişlərə verilən qiymətləndirmə ilə gecikmiş sifarişlərin qiymətləndirilməsinin ortalamasına baxırıq ..Qiymətləndirmənin səbəbi olmasada təsir etdiyi görülür 

In [28]:

import pandas as pd

df = pd.read_sql("""
    SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Gecikib'
        ELSE 'Vaxtında'
        END AS delivery_status,
        AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r
    ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Gecikib'
        ELSE 'Vaxtında'
        END """, engine)

df

,delivery_status,avg_review_score
0,Vaxtinda,4.293718
1,Gecikib,2.566494


Verilən 1 və 2 qiymətləndirmələrin neçə faizinin gecikməyə görə olduğuna baxırıq və görürükki 
vaxtında çatdırılma təxminən hər 100 rəydən 9-u narazıdır.
Gecikmiş çatdırılma təxminən hər 100 rəydən 54-ü narazıdır və buda az verilən qiymətləndirmənin böyük bir hissəsinin hardasa yarısı qədər gecikmənin payında düşür   

In [29]:
df = pd.read_sql("""
    SELECT CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtinda'
        ELSE 'Gecikib'
        END AS delivery_status,
        COUNT(*) AS total_reviews,
        AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100
            AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY
        CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtinda'
        ELSE 'Gecikib'
        END 
""", engine)

df

,delivery_status,total_reviews,unhappy_review_pct
0,Vaxtinda,89944,9.2791
1,Gecikib,6409,62.4122


Təxmini və faktiki çatdırılma tarixlərini müqayisə etdik. Təqvim günləri üzrə gecikmə qaydasını tətbiq edərək vaxtında çatdırılan və gecikən sifarişləri ayırdıq. Daha sonra bu qruplar üzrə müştərilərin orta rəy balını və narazı rəylərin faizini hesabladıq. Məqsədimiz çatdırılma gecikmələrinin müştəri məmnuniyyəti ilə əlaqəsini araşdırmaqdır.Bu nəticədən görə bilərik ki gecikmə vaxtı artdıqca aşağı rəylərdə artaraq davam edir.  

In [30]:
df = pd.read_sql("""
    SELECT CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtında'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 1 AND 3
        THEN '1-3 gun gecikib'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 4 AND 7
        THEN '4-7 gun gecikib'
        ELSE '8+ gun gecikib'
        END AS delay_bucket,
        COUNT(*) AS total_reviews,
        AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100
        AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) <= 0
        THEN 'Vaxtında'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 1 AND 3
        THEN '1-3 gun gecikib'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 4 AND 7
        THEN '4-7 gun gecikib'
        ELSE '8+ gun gecikib'
        END
""", engine)

df

,delay_bucket,total_reviews,unhappy_review_pct
0,4-7 gun gecikib,1756,67.6537
1,Vaxtinda,89944,9.2791
2,8+ gun gecikib,2797,79.1919
3,1-3 gun gecikib,1856,32.1659


## B2

In [31]:
df = pd.read_sql("""
    SELECT TOP 10 *
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df

,seller_id,zip_code_prefix,city,state,total_orders,items_sold,total_sales,total_freight
0,4869f7a5dfa277a7dca6462dcf3b52b2,14840,guariba,SP,1124,1148,226987.93,20019.13
1,53243585a1d6dc2643021fd1853d8905,42738,lauro de freitas,BA,348,400,217940.44,12856.58
2,4a3ca9315b744ce9f8e9374361493884,14940,ibitinga,SP,1772,1949,196882.12,34338.31
3,fa1c13f2614d7b5c4749cbc52fecda94,13170,sumare,SP,578,579,190917.14,9916.36
4,7c67e1448b00f6e969d365cea6b010ab,8577,itaquaquecetuba,SP,973,1355,186570.05,51236.64
5,7e93a43ef30c4f03f38b393420bc753a,6429,barueri,SP,319,322,165981.49,5992.06
6,da8622b14eb17ae2831f4ac5b9dab84a,13405,piracicaba,SP,1311,1548,159816.87,24889.91
7,7a67c85e85bb2ce8582c35f2203ad736,3426,sao paulo,SP,1145,1155,139658.69,20619.83
8,1025f0e2d44d7041d6cf58b6550e0bfa,3204,sao paulo,SP,910,1420,138208.56,33716.40
9,955fee9216a65b617aa5c0531780ce60,4782,sao paulo,SP,1261,1472,131836.71,24769.77


Biz bu kodda satıcıların hər sifarişinə orta hesabla nə qədər satış məbləği düşdüyünü hesablamaq və satıcıları müqayisə etdik.Ümumi satış məbləğini ümumi sifariş sayına böldük və ən yüksək orta satış göstəricisinə malik satıcıları müəyyən etdik. Amma ilk yerlərdə yalnız bir neçə sifarişi olan satıcılar da var. Buna görə onları çoxsaylı sifariş qəbul edən satıcılarla birbaşa müqayisə etmək düzgün olmaya bilər.

In [32]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY avg_sales_per_order DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,e3b4998c7a498169dc7bce44e6bb6277,1,6735.00,6735.000000
1,80ceebb4ee9b31afb6c6a916a574a1e2,1,6729.00,6729.000000
2,ee27a8f15b1dded4d213a468ba4eb391,1,6499.00,6499.000000
3,b37c4c02bda3161a7546a4e6d222d5b2,4,24075.00,6018.750000
4,585175ec331ea177fa47199e39a6170a,1,3549.00,3549.000000
5,abe021b01ba992245271b9aa422032df,2,6720.00,3360.000000
6,a00824eb9093d40e589b940ec45c4eb0,3,9399.97,3133.323333
7,e2a1ac9bf33e5549a2a4f834e70df2f8,5,14999.45,2999.890000
8,e908c0f3646e8b60375734a350d95d71,1,2951.00,2951.000000
9,d63c73efd41eb002280e7ec831424edb,2,5598.00,2799.000000


Ən azı 100 sifarişi olan satıcılar siyahısına baxırıqki bu az sifarişli satıcıların yüksək orta göstəricilərinin müqayisəyə təsirini azaldacaq. Onlar arasında birinci satıcının sifariş başına orta satış məbləği ən yüksək idi 626,27 və 348 sifarişlədir amma bu satıcının daha çox mənfəət qazandığını sübut etmir. 

In [33]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    WHERE total_orders >= 100
    ORDER BY avg_sales_per_order DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,53243585a1d6dc2643021fd1853d8905,348,217940.44,626.265632
1,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,520.318150
2,edb1ef5e36e0c8cd84eb3c9b003e486d,165,79144.90,479.666061
3,f5a590cf36251cf1162ea35bef76fe84,113,51232.78,453.387434
4,ccc4bbb5f32a6ab2b7066a4130f114e3,184,72926.72,396.340870
5,25c5c91f63607446a97b143d2d535d31,156,54208.78,347.492179
6,5dceca129747e92ff8ef7a997dc4f8ca,322,111126.73,345.114068
7,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,330.306471
8,522620dcb18a6b31cd7bdf73665113a9,172,56690.49,329.595872
9,17a053fcb14bd219540cbde0df490be0,124,40353.24,325.429355


Ən çox sifarişi olan ilk 10 satıcını müəyyənləşdiririk

In [34]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_orders DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,6560211a19b47992c3666cc44a7e94c0,1819,120702.83,66.356696
1,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,111.107291
2,cc419e0650a3c5ba77189a1882b7556a,1651,101090.96,61.230139
3,1f50f920176fa81dab994f9023523100,1399,106655.71,76.237105
4,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,121.904554
5,955fee9216a65b617aa5c0531780ce60,1261,131836.71,104.549334
6,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,121.972655
7,ea8482cd71df3c1969d7b9473ff13abc,1132,36696.76,32.417633
8,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,201.946557
9,3d871de0142ce09b7081e2b9d1733cb1,1064,93407.40,87.788910


Ən çox satışı olan ilk 10 satıcını müəyyənləşdiririk

In [35]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,201.946557
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,626.265632
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,111.107291
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,330.306471
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,191.747225
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,520.318150
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,121.904554
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,121.972655
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,151.877538
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,104.549334


Burada axtaririq ən yüksək ümumi satış edən satıcıların daşınma məbləği də yüksəkdirmi?  satış və daşınma göstəricilərinin necə dəyişdiyini müqayisə etmək üçün baxırıq.
3-cü satıcının ümumi daşınma məbləği ilk üçlükdə ən yüksəkdir 34.338,31

In [36]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,avg_sales_per_order
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,201.946557
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,626.265632
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,111.107291
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,330.306471
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,191.747225
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,520.318150
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,121.904554
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,121.972655
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,151.877538
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,104.549334


Cədvəldə satıcıları ümumi satış məbləğinə görə sıralayıb sonra hər satıcı üçün bir sifarişə düşən orta daşınma məbləğini hesablayırıq.

1-ci satıcı Ən yüksək ümumi satışa malikdir və orta daşınma məbləği 17.81.
2-ci satıcı Satış məbləği birinci satıcıya yaxındır, amma orta daşınma məbləği 36.94.
5-ci satıcı İlk 10 satıcı arasında orta daşınma məbləği ən yüksək olan satıcıdır 52.66.

In [37]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_freight AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_freight_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,avg_freight_per_order
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,17.810614
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,36.944195
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,19.378279
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,17.156332
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,52.658417
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,18.783887
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,18.985439
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,18.008585
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,37.050989
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,19.642958


In [38]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_freight AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_freight_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_orders DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,avg_freight_per_order
0,6560211a19b47992c3666cc44a7e94c0,1819,120702.83,27348.11,15.034695
1,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,19.378279
2,cc419e0650a3c5ba77189a1882b7556a,1651,101090.96,24846.03,15.049079
3,1f50f920176fa81dab994f9023523100,1399,106655.71,35057.23,25.058778
4,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,18.985439
5,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,19.642958
6,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,18.008585
7,ea8482cd71df3c1969d7b9473ff13abc,1132,36696.76,17291.87,15.275504
8,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,17.810614
9,3d871de0142ce09b7081e2b9d1733cb1,1064,93407.40,22108.25,20.778430


Bu analiz satıcılar üzrə çatdırılma xərclərinin satış məbləğinə nisbətini müqayisə etməyə imkan verir. Göstəricinin yüksək olması çatdırılma xərcinin satışa nisbətən daha böyük pay tutduğunu göstərir. Aşağı olması isə çatdırılma xərcinin satışa nisbətən daha kiçik pay tutduğunu göstərir.

In [39]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_freight AS DECIMAL(18, 2)) * 100
            / NULLIF(total_sales, 0) AS freight_to_sales_pct
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,freight_to_sales_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,8.819469
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,5.899125
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,17.441051
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,5.194065
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,27.462414
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,3.610077
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,15.574019
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,14.764445
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,24.395305
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,18.788219


Satıcıların performansının təhlili göstərdi ki, ümumi satış məbləği, sifariş sayı və çatdırılma xərclərinin satışdakı payı satıcılar arasında fərqlənir. Bəzi satıcılar yüksək satış gəliri əldə etdiyi halda, çatdırılma xərclərinin satışdakı payı nisbətən aşağıdır. Digər satıcılarda isə bu pay daha yüksəkdir və çatdırılma xərclərinin əlavə araşdırılmasına ehtiyac ola bilər. Bu təhlil satıcıların satış göstəricilərini müqayisə etməyə və çatdırılma xərclərinə diqqət tələb edən halları müəyyənləşdirməyə kömək edir.

## B3

In [42]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.order_items
""", engine)

df

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
